# 02b — Signal density with a known reference (NCE)

An isolated experiment before continuing notebook 02. Train **one signal candidate** with balanced binary cross entropy and logit
$$
\ell_\psi(x)=f_\psi(x)-\log m(x).
$$
The saved reference flow is frozen. Training noise is now $m=0.8q_\phi+0.2g_{\mathrm{broad}}$, where the broad Gaussian has twice the fitted envelope widths. The NN models the signal log density rather than learning both sides of the ratio. With equal class priors the optimal logit is $\log p_S-\log m$.

We use an integrable parameterization: $f_\psi=\log g + 30\tanh(h_\psi/30)+c$, where $g$ is a diagonal Gaussian fitted only to optimization signal events. The four-layer, 1024-wide SiLU network learns a bounded correction; $c$ is a learned scalar normalization offset. This is also an architectural change from the baseline classifier, so a successful result would not isolate the effect of the known-reference term alone. Finite integrability does **not** guarantee accurate importance-sampling normalization or correct far tails.

Reuse the current TAG, saved signal sample and reference flow. No background model is required. Nothing is installed into the main hybrid model. Candidate files go under `hybrid/signal_nce/known_mixture_gaussian_v2/`. A GPU and sufficient host RAM for the 20M-event pool are recommended. The full training is substantial; interrupted training resumes from the last completed epoch.

This new experiment addresses the artificial tail mass found in v1. Both signal and noise events use **log m** during training. Downstream signal/reference diagnostics still use **f - log q**, so their meaning and baseline comparisons are unchanged. The old candidate remains in its original directory.


In [ ]:
# Keep the existing run: this experiment has its own output directory.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_10m_slowlr_v3_20261001"  # Fresh run: 10M nominal-ratio events per class and gradual LR decay.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; verify that it points to the intended run.")
print("Use a fresh kernel when switching between the old exercises and calibration.")

In [ ]:

import gc
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import logsumexp
from IPython.display import display
from flow_reference import ReferenceFlow
from sampling import read_sample
from signal_nce import SignalNCE, train_signal, log_density, signal_envelope
from signal_mixture_noise import mixture_log_prob, sample_mixture, broad_monitor

EXPERIMENT = 'known_mixture_gaussian_v2'
NOISE = dict(broad_fraction=0.2, width_scale=2.0)
MONITOR = dict(n=100_000, seed=7_202_812, width_scale=2.0, every=5)
TRAIN = True  # False: load the saved best candidate and rerun diagnostics only.
N_PER_CLASS = 10_000_000  # Pool size, before the 60/15/25 split.
N_DIAGNOSTIC = 1_000_000
N_NORMALIZATION = 1_000_000
DIAGNOSTIC_SEED = 7_212_610  # Fresh final banks, independent of training and monitor.
SETTINGS = dict(seed=7202610, epochs=150, batch_size=4096,
                width=1024, layers=4, bound=30., learning_rate=1e-3)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = RUN / 'hybrid'
OUT = ROOT / 'signal_nce' / EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)
REFERENCE = ROOT / 'reference.pt'
SAMPLE = RUN / 'samples' / 'signal.parquet'
for path in [REFERENCE, SAMPLE, RUN / 'config.json', ROOT / 'ratios/signal/ensemble.json']:
    if not path.is_file():
        raise FileNotFoundError(f'Required existing artifact missing: {path}')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(2**20), b''):
            h.update(block)
    return h.hexdigest()

PROVENANCE = dict(reference_sha256=sha256_file(REFERENCE),
                  config_sha256=sha256_file(RUN / 'config.json'),
                  trainer_sha256=sha256_file(SOURCE / 'signal_nce.py'),
                  mixture_helper_sha256=sha256_file(SOURCE / 'signal_mixture_noise.py'),
                  monitor_helper_sha256=sha256_file(SOURCE / 'signal_tail_diagnostics.py'),
                  noise=NOISE, monitor=MONITOR,
                  sample_path=str(SAMPLE), sample_bytes=SAMPLE.stat().st_size,
                  sample_mtime_ns=SAMPLE.stat().st_mtime_ns,
                  noise_seed=SETTINGS['seed'] + 101)
flow = ReferenceFlow.load(REFERENCE, DEVICE)
flow.flow.eval()
for parameter in flow.flow.parameters():
    parameter.requires_grad_(False)
print('Device:', DEVICE, '\nCandidate:', OUT)
print('The saved baseline and reference will not be overwritten.')

## Train or resume

10M signal events from the existing `ratio_train` partition and 10M negative events: 8M frozen-flow draws and 2M draws from the twice-width Gaussian. Component rows are shuffled before splitting. The mixture and candidate envelopes use the same optimization-only signal fit. Each class has 6M optimization, 1.5M validation and 2.5M reserved events. The Gaussian envelope uses only optimization events. Every batch is balanced, so no class-prior correction is required. NAdam starts at $10^{-3}$; StepLR halves it every ten epochs through 150 epochs (about $6.1\times10^{-8}$ in epochs 141–150). Checkpoint selection uses validation BCE only.

The cached arrays include mixture log densities on both classes, avoiding repeated flow evaluation. Cache metadata must match exactly; use a new experiment name when changing settings or inputs. Rerunning with completed checkpoints skips optimization. The reserved pool is not used for model selection; the comparisons below use independently generated events.

A fixed independent 100k-event broad-Gaussian bank monitors the unnormalized signal integral, ESS and largest weight share at epoch 1 and every five epochs. It never enters the training gradients or checkpoint ranking; the best checkpoint still uses validation BCE alone. This monitor is a warning, not certification. Fresh, larger final integration banks remain the acceptance check. Training/validation BCE is now against **m**, and should not be compared numerically with the old q-based training history.


In [ ]:
if TRAIN:
    cache = OUT / 'training_cache'
    cache.mkdir(exist_ok=True)
    cache_identity = dict(provenance=PROVENANCE, n_per_class=N_PER_CLASS)
    marker = cache / 'complete.json'
    paths = {name: cache / f'{name}.npy' for name in ['signal', 'noise', 'logm_signal', 'logm_noise']}
    if marker.exists():
        if json.loads(marker.read_text()) != cache_identity:
            raise ValueError('Cache provenance changed. Choose a new EXPERIMENT.')
        if not all(p.is_file() for p in paths.values()) or not (cache / 'envelope.json').is_file():
            raise FileNotFoundError('Incomplete cache: remove complete.json and rerun this cell.')
    else:
        x = read_sample(RUN, 'signal', partition='ratio_train', n=N_PER_CLASS)
        np.save(paths['signal'], x)
        envelope_mean, envelope_std = signal_envelope(x, SETTINGS['seed'])
        np.save(paths['logm_signal'], mixture_log_prob(x, flow, envelope_mean, envelope_std, **NOISE))
        (cache / 'envelope.json').write_text(json.dumps(dict(mean=envelope_mean.tolist(), std=envelope_std.tolist()), indent=2))
        del x
        gc.collect()
        x = sample_mixture(N_PER_CLASS, flow, envelope_mean, envelope_std, PROVENANCE['noise_seed'], **NOISE)
        np.save(paths['noise'], x)
        np.save(paths['logm_noise'], mixture_log_prob(x, flow, envelope_mean, envelope_std, **NOISE))
        del x
        gc.collect()
        marker.write_text(json.dumps(cache_identity, indent=2))
    arrays = {k: np.load(p, mmap_mode='r') for k, p in paths.items()}
    envelope = json.loads((cache / 'envelope.json').read_text())
    monitor = broad_monitor(np.array(envelope['mean']), np.array(envelope['std']), **MONITOR)
    candidate, history = train_signal(arrays['signal'], arrays['noise'],
                                      arrays['logm_signal'], arrays['logm_noise'],
                                      OUT, SETTINGS, PROVENANCE, device=DEVICE, monitor=monitor)
    del arrays
    gc.collect()
else:
    checkpoint = torch.load(OUT / 'best.pt', map_location=DEVICE, weights_only=False)
    identity = checkpoint['identity']
    if (identity['provenance'] != PROVENANCE or identity['settings'] != SETTINGS
            or identity['n_per_class'] != N_PER_CLASS):
        raise ValueError('Saved candidate does not match the requested experiment.')
    candidate = SignalNCE(**checkpoint['architecture']).to(DEVICE)
    candidate.load_state_dict(checkpoint['model'])
    candidate.eval()
    history = json.loads((OUT / 'history.json').read_text())
    print('Loaded best epoch:', checkpoint['epoch'])

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUT / 'history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_frame.plot(x='epoch', y=['train_bce', 'val_bce'], ax=axes[0])
history_frame.plot(x='epoch', y='learning_rate', logy=True, ax=axes[1])
fig.tight_layout()
fig.savefig(OUT / 'training.png', dpi=150)
plt.show()

monitor_rows = history_frame.dropna(subset=['broad_log_mean']) if 'broad_log_mean' in history_frame else pd.DataFrame()
if not monitor_rows.empty:
    display(monitor_rows[['epoch', 'broad_mean', 'broad_mc_se', 'broad_ess', 'broad_largest_weight_fraction']])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(monitor_rows.epoch, monitor_rows.broad_log_mean, marker='o')
    ax.axhline(0, color='k', linestyle='--')
    ax.set(xlabel='Epoch', ylabel='Log estimated signal integral under broad proposal')
    fig.tight_layout()
    fig.savefig(OUT / 'broad_training_monitor.png', dpi=150)
    plt.show()


## Fresh comparison: candidate, baseline ensemble and exact oracle

These events are independent of optimization and validation. This common-reference comparison uses $f-\log q$ for BCE and classifier calibration; the training logit is $f-\log m$. Compare paired BCE differences, exact log-ratio errors, and the populated tail bins together. A visually better reliability curve alone does not establish a better density. Approximate pointwise intervals can be unreliable in sparse bins; inspect their counts.

Reports are generated separately: the candidate is **never averaged with the baseline**. The exact simulator density appears for the first time here. It cannot affect checkpoint selection. Once these diagnostics guide development, use a new diagnostic seed for final validation.

In [ ]:
from sampling import sample_process, process_log_density
from utils import RatioEnsemble
from ratio_diagnostics import diagnostic_report

EVAL = OUT / f'diagnostics_seed_{DIAGNOSTIC_SEED}'
EVAL.mkdir(exist_ok=True)
baseline = RatioEnsemble.load(ROOT / 'ratios/signal', device=DEVICE)
x_num = sample_process('signal', N_DIAGNOSTIC, np.random.default_rng(DIAGNOSTIC_SEED))
x_den = flow.sample(N_DIAGNOSTIC, DIAGNOSTIC_SEED + 1, batch_size=8192)
q_num = flow.log_prob(x_num, batch_size=8192).astype(np.float64)
q_den = flow.log_prob(x_den, batch_size=8192).astype(np.float64)
exact_num = process_log_density(x_num, 'signal') - q_num
exact_den = process_log_density(x_den, 'signal') - q_den
candidate_num = log_density(candidate, x_num) - q_num
candidate_den = log_density(candidate, x_den) - q_den
member_num = baseline.member_log_ratios(x_num)
member_den = baseline.member_log_ratios(x_den)
baseline_num = logsumexp(member_num, axis=0) - np.log(len(member_num))
baseline_den = logsumexp(member_den, axis=0) - np.log(len(member_den))
score_edges = np.unique(np.round(np.r_[np.linspace(0, 1, 26), np.arange(.70, .851, .01)], 8))
summaries = []
models_to_report = [('baseline', member_num, member_den),
                    ('candidate', candidate_num[None, :], candidate_den[None, :])]
previous_path = ROOT / 'signal_nce/known_q_gaussian_v1/best.pt'
if previous_path.is_file():
    previous_state = torch.load(previous_path, map_location=DEVICE, weights_only=False)
    if previous_state['identity']['provenance']['reference_sha256'] != PROVENANCE['reference_sha256']:
        raise ValueError('Previous candidate used a different reference flow.')
    previous_candidate = SignalNCE(**previous_state['architecture']).to(DEVICE)
    previous_candidate.load_state_dict(previous_state['model'])
    previous_num = log_density(previous_candidate, x_num)-q_num
    previous_den = log_density(previous_candidate, x_den)-q_den
    models_to_report.append(('previous_candidate', previous_num[None, :], previous_den[None, :]))
    del previous_candidate, previous_state
else:
    print('Previous candidate checkpoint absent; baseline comparison is still available.')
for label, ln, ld in models_to_report:
    report = diagnostic_report(ln, ld, exact_num, exact_den,
                               title=f'Signal: {label}', score_edges=score_edges)
    report['summary'].to_csv(EVAL / f'{label}_summary.csv', index=False)
    report['bins'].to_csv(EVAL / f'{label}_bins.csv', index=False)
    chosen = report['summary'].query("model == 'Ensemble'").copy()
    chosen['model'] = label
    summaries.append(chosen)
    display(report['summary'])
    for name, fig in report['figures']:
        fig.savefig(EVAL / f'{label}_{name}.png', dpi=150, bbox_inches='tight')
        display(fig)
        plt.close(fig)
comparison = pd.concat(summaries, ignore_index=True)
comparison.to_csv(EVAL / 'comparison.csv', index=False)
display(comparison)

# Same-event pairing is essential: this error bar concerns evaluation statistics only.
delta_num = np.logaddexp(0, -candidate_num) - np.logaddexp(0, -baseline_num)
delta_den = np.logaddexp(0, candidate_den) - np.logaddexp(0, baseline_den)
difference = .5 * (delta_num.mean() + delta_den.mean())
se = .5 * np.sqrt(delta_num.var(ddof=1)/N_DIAGNOSTIC + delta_den.var(ddof=1)/N_DIAGNOSTIC)
paired = dict(candidate_minus_baseline_bce=float(difference), se=float(se),
              low95=float(difference-1.96*se), high95=float(difference+1.96*se))
(EVAL / 'paired_bce.json').write_text(json.dumps(paired, indent=2))
print('Negative favors candidate; this interval excludes training uncertainty:', paired)

## Independent normalization and tail stress check

Estimate $Z=E_q[\exp(f-\log q)]$ on a **separate** reference bank, then check $E_q[r/Z]$ on the diagnostic denominator bank above. The latter was not used to estimate $Z$. Also record ESS, the largest normalized weight and the fraction of total weight carried by the largest 0.1% of events. A small Monte Carlo SE is not proof that unsampled tails are harmless.

The candidate's normalized log density is $f-\log Z$. This optional scalar correction is saved separately and is not applied retroactively to raw classifier calibration. Both candidate and baseline use the same integration bank for this comparison. No candidate is promoted automatically.

In [ ]:
normalization_x = flow.sample(N_NORMALIZATION, DIAGNOSTIC_SEED + 2, batch_size=8192)
normalization_q = flow.log_prob(normalization_x, batch_size=8192).astype(np.float64)
normalization_members = baseline.member_log_ratios(normalization_x)
normalization_logits = {
    'candidate': log_density(candidate, normalization_x) - normalization_q,
    'baseline': logsumexp(normalization_members, axis=0) - np.log(len(normalization_members)),
}
check_logits = {'candidate': candidate_den, 'baseline': baseline_den}
rows = []
for name, values in normalization_logits.items():
    logz = float(logsumexp(values) - np.log(len(values)))
    scaled = np.exp(values - logz)
    relative_se = float(scaled.std(ddof=1) / np.sqrt(len(scaled)))
    check = check_logits[name] - logz
    logmean = logsumexp(check) - np.log(len(check))
    normalized_weights = np.exp(check - logsumexp(check))
    k = max(1, int(np.ceil(.001 * len(check))))
    rows.append(dict(model=name, log_Z=logz, Z_relative_mc_se=relative_se,
                     independent_mean_ratio=float(np.exp(logmean)),
                     independent_mean_ratio_se=float(np.exp(logmean) *
                         np.exp(check-logmean).std(ddof=1) / np.sqrt(len(check))),
                     independent_ess=float(1 / np.square(normalized_weights).sum()),
                     maximum_normalized_ratio=float(np.exp(check.max())),
                     top_0p1_percent_weight_fraction=float(np.partition(normalized_weights, -k)[-k:].sum())))
normalization = pd.DataFrame(rows)
display(normalization)
normalization.to_csv(EVAL / 'normalization.csv', index=False)
(EVAL / 'normalization.json').write_text(json.dumps(dict(
    reference_sha256=PROVENANCE['reference_sha256'],
    checkpoint_sha256=sha256_file(OUT / 'best.pt'),
    integration_seed=DIAGNOSTIC_SEED+2, integration_size=N_NORMALIZATION,
    diagnostic_size=N_DIAGNOSTIC, results=rows), indent=2))
print('Ready for review:', EVAL)
print('Review comparison.csv, paired_bce.json, calibration/exact-error figures and normalization.csv together.')

## Investigate the extreme weights — no retraining

This experiment defaults to training once, then checking fresh diagnostic banks. For later diagnostics-only reruns, set `TRAIN = False` and keep this experiment name and diagnostic seed. The v1 extreme-event check below reuses the old saved coordinates explicitly. All checks below use the saved best candidate. They do not change model files, clip ratios, or replace normalization constants. Different hardware or library versions can change the sampled events; we save the selected coordinates explicitly.

We inspect the union of the 20 largest candidate, baseline and **exact** ratios in each of the two reference banks. The decomposition
$$
\log r_{\mathrm{candidate}}-\log r_{\mathrm{exact}}=f_\psi-\log p_S
$$
removes the shared reference term. It tells us whether the candidate overpredicts the signal density at the offending event. Large exact weights also point toward reference coverage or reference numerics. Table means below use **raw** ratios: the exact ratio should have population mean one. Monte Carlo standard errors and ESS can be misleading when rare events dominate.

In [ ]:
from signal_tail_diagnostics import (weight_summary, gaussian_log_density,
                                     tail_table, precision_probe)
import importlib.metadata

TOP_K = 20
TAIL = EVAL / 'tail_investigation'
TAIL.mkdir(exist_ok=True)
g_mean = candidate.mean.detach().cpu().numpy().astype(np.float64)
g_std = candidate.std.detach().cpu().numpy().astype(np.float64)
exact_normalization = process_log_density(normalization_x, 'signal') - normalization_q
banks = {
    'diagnostic_q': dict(x=x_den, q=q_den, candidate=candidate_den,
                         baseline=baseline_den, exact=exact_den),
    'normalization_q': dict(x=normalization_x, q=normalization_q,
                            candidate=normalization_logits['candidate'],
                            baseline=normalization_logits['baseline'], exact=exact_normalization),
}
tables, summaries = [], []
for bank_name, bank in banks.items():
    table = tail_table(bank_name, bank['x'], bank['q'], bank['candidate'],
                       bank['baseline'], bank['exact'], g_mean, g_std, TOP_K)
    tables.append(table)
    np.savez_compressed(TAIL / f'{bank_name}_extreme_events.npz',
                        event_index=table.event_index.to_numpy(),
                        x=bank['x'][table.event_index.to_numpy()])
    for name in ['candidate', 'baseline', 'exact']:
        v = bank[name]
        summary = weight_summary(v)
        # Influence diagnostic only: removing events is NOT a normalization fix.
        biggest = int(np.argmax(v))
        remainder = np.delete(v, biggest)
        summary['remaining_contribution_after_largest_to_full_bank_mean'] = float(
            np.exp(logsumexp(remainder)-np.log(len(v))))
        summaries.append(dict(bank=bank_name, model=name, **summary))
tails = pd.concat(tables, ignore_index=True)
tails.to_csv(TAIL / 'extreme_events.csv', index=False)
weights = pd.DataFrame(summaries)
weights.to_csv(TAIL / 'reference_bank_weights.csv', index=False)
with pd.option_context('display.max_columns', None, 'display.width', 240):
    display(weights)
    for bank_name in banks:
        print('Largest candidate weights:', bank_name)
        display(tails[tails.bank.eq(bank_name)].head(10))
# Plots explicitly retain the single-event contribution.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (bank_name, bank) in zip(axes, banks.items()):
    for name in ['candidate', 'baseline', 'exact']:
        v = bank[name]
        fraction = np.exp(np.sort(v)[::-1]-logsumexp(v))
        counts = np.unique(np.geomspace(1, len(v), 300).astype(int))
        ax.plot(counts, np.cumsum(fraction)[counts-1], label=name)
    ax.set(xscale='log', xlabel='Number of largest-weight events',
           ylabel='Cumulative fraction of total weight', title=bank_name)
    ax.legend()
fig.tight_layout()
fig.savefig(TAIL / 'weight_concentration.png', dpi=150)
plt.show()
manifest = dict(diagnostic_seed=DIAGNOSTIC_SEED, normalization_seed=DIAGNOSTIC_SEED+2,
                n_diagnostic=N_DIAGNOSTIC, n_normalization=N_NORMALIZATION,
                sampling_batch_size=8192, top_k=TOP_K, device=DEVICE,
                torch=str(torch.__version__), numpy=np.__version__,
                nflows=importlib.metadata.version('nflows'),
                candidate_sha256=sha256_file(OUT / 'best.pt'),
                reference_sha256=PROVENANCE['reference_sha256'],
                tail_helper_sha256=sha256_file(SOURCE / 'signal_tail_diagnostics.py'))
(TAIL / 'manifest.json').write_text(json.dumps(manifest, indent=2))

## Reference precision and invertibility at the same events

Run CPU float32 and float64 copies of the saved models on the **same physical coordinates**. Compare with the original device's float32 values. Check the standardized-coordinate round trip $x\to z\to x$ and cancellation of the forward/inverse log Jacobians. A separate 512-event generative check retains its original latent coordinates and compares the sampling and reevaluated log densities.

These are numerical probes, not corrected production weights. A large precision change is evidence to investigate; good round trips alone do not prove the reference is accurate. Promoting stored float32 coordinates cannot recover precision lost when those events were generated. The original models are never converted in place. All selected events are saved even if a precision probe raises an error.

In [ ]:
precision_tables, generative_tables, precision_errors = [], [], []
for bank_name, bank in banks.items():
    selected = tails[tails.bank.eq(bank_name)].reset_index(drop=True)
    points = bank['x'][selected.event_index.to_numpy()]
    try:
        probe, generative = precision_probe(flow, candidate, points)
    except (RuntimeError, ValueError, AssertionError) as error:
        precision_errors.append(dict(bank=bank_name, error=repr(error)))
        print('Precision probe failed; selected events were saved:', bank_name, repr(error))
        continue
    combined = pd.concat([selected, probe], axis=1)
    combined['logq_cpu32_minus_original'] = combined.log_q_cpu32-combined.log_q32
    combined['f_cpu32_minus_original'] = combined.f_cpu32-combined.f_candidate32
    combined['exact_logr_cpu64'] = combined.log_p_exact-combined.log_q_cpu64
    combined['candidate_minus_exact_logp_cpu64'] = combined.f_cpu64-combined.log_p_exact
    precision_tables.append(combined)
    generative.insert(0, 'bank_probe', bank_name)
    generative_tables.append(generative)
    columns = ['bank', 'event_index', 'candidate_log_ratio', 'exact_log_ratio',
               'candidate_minus_exact_logp', 'logq_cpu32_minus_original',
               'delta_logq_64_minus_32', 'delta_f_64_minus_32',
               'candidate_logr_cpu64', 'exact_logr_cpu64',
               'x_roundtrip_max_std32', 'x_roundtrip_max_std64',
               'logdet_roundtrip_abs32', 'logdet_roundtrip_abs64']
    with pd.option_context('display.max_columns', None, 'display.width', 240):
        display(combined[columns].head(10))
    gc.collect()
if precision_tables:
    pd.concat(precision_tables, ignore_index=True).to_csv(TAIL / 'precision_extreme_events.csv', index=False)
    generative_checks = pd.concat(generative_tables, ignore_index=True)
    generative_checks.to_csv(TAIL / 'generative_roundtrip.csv', index=False)
    display(generative_checks)
(TAIL / 'precision_errors.json').write_text(json.dumps(precision_errors, indent=2))

## Independent integration with the Gaussian envelope

Draw from the fitted envelope $g$ instead of the flow, and estimate
$$Z_\psi=E_g[\exp(f_\psi-\log g)].$$
This avoids reference-flow sampling and density evaluation entirely. The exact control $E_g[p_S/g]=1$ tests the same integration proposal. Run three independent banks and report each separately, including ESS and the largest weight fraction. The bounded correction guarantees finite weights here but permits a very large range; agreement of a few banks is evidence, not a proof of tail coverage.

We also evaluate $Z_\psi$ under a broader Gaussian with twice the standard deviations as a stress test for mass outside the fitted envelope's typical region. Each bank uses its **own actual proposal density** in the denominator. Neither proposal changes the candidate or uses the analytic target to train it. Results are saved as diagnostics only; no main normalization is overwritten.

In [ ]:
N_GAUSSIAN = 1_000_000
GAUSSIAN_REPEATS = 3
GAUSSIAN_SEED = DIAGNOSTIC_SEED + 100
proposal_rows = []
for proposal_index, scale in enumerate([1., 2.]):
    for repeat in range(GAUSSIAN_REPEATS):
        seed = GAUSSIAN_SEED + 10*proposal_index + repeat
        rng = np.random.default_rng(seed)
        # Retain double coordinates for the analytic proposal and oracle.
        # The saved candidate is evaluated in its original float32 precision.
        points = rng.normal(size=(N_GAUSSIAN, len(g_mean))) * (scale*g_std) + g_mean
        proposal_logp = gaussian_log_density(points, g_mean, scale*g_std)
        candidate_logw = log_density(candidate, points) - proposal_logp
        exact_logw = process_log_density(points, 'signal') - proposal_logp
        for name, values in [('candidate', candidate_logw), ('exact', exact_logw)]:
            proposal_rows.append(dict(proposal_std_scale=scale, repeat=repeat,
                                      seed=seed, model=name, **weight_summary(values)))
        # Save the largest candidate-weight events from every alternative bank.
        indices = np.argsort(candidate_logw)[-TOP_K:][::-1]
        frame = pd.DataFrame(points[indices], columns=[f'x{i+1}' for i in range(len(g_mean))])
        frame['event_index'] = indices
        frame['log_proposal'] = proposal_logp[indices]
        frame['candidate_log_weight'] = candidate_logw[indices]
        frame['exact_log_weight'] = exact_logw[indices]
        frame['candidate_minus_exact_logp'] = candidate_logw[indices]-exact_logw[indices]
        frame.to_csv(TAIL / f'gaussian_scale_{scale:g}_seed_{seed}_extremes.csv', index=False)
        pd.DataFrame(proposal_rows).to_csv(TAIL / 'gaussian_normalization.csv', index=False)
        print(f'Finished Gaussian proposal scale={scale:g}, bank={repeat+1}/{GAUSSIAN_REPEATS}', flush=True)
        del points, proposal_logp, candidate_logw, exact_logw
        gc.collect()
proposal_results = pd.DataFrame(proposal_rows)
with pd.option_context('display.max_columns', None, 'display.width', 240):
    display(proposal_results)
manifest.update(gaussian_seed=GAUSSIAN_SEED, n_gaussian=N_GAUSSIAN,
                gaussian_repeats=GAUSSIAN_REPEATS, gaussian_std_scales=[1., 2.])
(TAIL / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print('Review extreme_events.csv, precision_extreme_events.csv, reference_bank_weights.csv,')
print('generative_roundtrip.csv and gaussian_normalization.csv in:', TAIL)

## Previously observed v1 extreme events

These saved coordinates are a development check, independent of the new fresh-bank summaries. They were not inserted into the training data. Compare the old and new signal-density error at the exact same points, including the event that dominated the old integral. Files are read only from the v1 directory. Missing files do not block fresh diagnostics.

In [ ]:
old_tail = ROOT / 'signal_nce/known_q_gaussian_v1/diagnostics_seed_7202610/tail_investigation'
challenge_tables = []
for bank_name in ['diagnostic_q', 'normalization_q']:
    saved = old_tail / f'{bank_name}_extreme_events.npz'
    if not saved.is_file():
        print('Previous event file unavailable:', saved)
        continue
    with np.load(saved) as archive:
        points = archive['x']
        indices = archive['event_index']
    exact_logp = process_log_density(points, 'signal')
    new_logp = log_density(candidate, points)
    frame = pd.DataFrame(dict(bank=bank_name, event_index=indices,
                              exact_logp=exact_logp, new_logp=new_logp,
                              new_minus_exact_logp=new_logp-exact_logp))
    old_csv = old_tail / 'extreme_events.csv'
    if old_csv.is_file():
        old = pd.read_csv(old_csv)
        frame = frame.merge(old[['bank', 'event_index', 'candidate_minus_exact_logp']],
                            on=['bank', 'event_index'], how='left', validate='one_to_one')
        frame = frame.rename(columns={'candidate_minus_exact_logp': 'old_minus_exact_logp'})
    challenge_tables.append(frame)
if challenge_tables:
    challenge = pd.concat(challenge_tables, ignore_index=True)
    challenge.to_csv(TAIL / 'previous_extreme_event_comparison.csv', index=False)
    with pd.option_context('display.max_columns', None):
        display(challenge[challenge.event_index.eq(866315)])
        display(challenge)


Interpret the checks together:

- **Large candidate/exact density disagreement at the extreme event:** candidate extrapolation contributes to the weight.
- **Both learned and exact ratios are huge:** investigate reference coverage and numerical density evaluation.
- **Large float32/float64 or round-trip differences:** investigate flow numerics before interpreting the physical tail.
- **Gaussian integration is stable near one while flow-bank estimates disagree:** evidence that flow-based integration is the immediate problem; inspect the exact controls and precision results to distinguish coverage from numerics.
- **Gaussian and broader-Gaussian results are also unstable or disagree:** the candidate's normalization remains unresolved. Increasing the bank size or deleting the largest event does not establish correctness.

These possibilities can coexist. The saved raw event coordinates allow targeted follow-up without another training run.

## Decision after this run

A promising candidate should improve fresh-event BCE and exact log-ratio errors without merely moving the problem into a less populated tail. Check the 0.70–0.85 region, event counts, extreme weights, ESS, and independent normalization. If the signals conflict or depend strongly on the integration seed, we should investigate before replacing the baseline. A single candidate does not measure seed-to-seed stability; an ensemble would be a subsequent experiment.

To rerun diagnostics, set `TRAIN = False`. To repeat an independent diagnostic bank, change `DIAGNOSTIC_SEED`. To train a different configuration, change `EXPERIMENT` so that the existing experiment remains available. Continue notebook 02 only after reviewing this signal comparison.